In [0]:
import dlt
from pyspark.sql import functions as F

print("Setup done")

In [0]:
silver = spark.read.table("capstone_project_dev.silver.validated_metadata")

In [0]:
total_tables  = silver.select("table_name").distinct().count()
total_columns = silver.select("table_name", "column_name").distinct().count()

# Structural compliance — tables with column_count >= threshold (mean - 1 stddev = 38.6)
structural_compliance_pct = (
    silver.select("table_name", "rule06_below_col_standard")
          .distinct()
          .filter(F.col("rule06_below_col_standard") == False)
          .count()
    / total_tables * 100
)

# PII coverage — PII rows correctly marked Confidential
pii_total = silver.filter(F.col("pii_flag") == True).count()
pii_covered = silver.filter(
    (F.col("pii_flag") == True) &
    (F.col("security_classification") == "Confidential")
).count()
pii_coverage_pct = (pii_covered / pii_total * 100) if pii_total > 0 else 0

# Certification coverage — distinct columns with non-null certification
cert_covered = silver.select("table_name", "column_name", "certification_level") \
    .distinct() \
    .filter(F.col("certification_level").isNotNull()) \
    .count()
cert_coverage_pct = (cert_covered / total_columns * 100)

# CDEs missing a steward
cde_missing_steward = silver.filter(
    (F.col("critical_data_element_flag") == True) &
    (F.col("data_steward").isNull())
).count()

# Metadata completeness — across 4 key fields, deduplicated
fields_to_check = ["column_desc", "term_name", "data_steward", "security_classification"]
total_possible  = total_columns * len(fields_to_check)
total_filled    = sum(
    silver.select("table_name", "column_name", f)
          .distinct()
          .filter(F.col(f).isNotNull())
          .count()
    for f in fields_to_check
)
metadata_completeness_pct = (total_filled / total_possible * 100)

# Overall compliance — deduplicated rows
compliant_rows = silver.select("table_name", "column_name", "compliance_flag") \
    .distinct() \
    .filter(F.col("compliance_flag") == "COMPLIANT") \
    .count()
non_compliant_rows = total_columns - compliant_rows
overall_compliance_pct = (compliant_rows / total_columns * 100)

print(f"Total tables              : {total_tables}")
print(f"Total columns             : {total_columns:,}")
print(f"Structural compliance %   : {structural_compliance_pct:.1f}%")
print(f"PII coverage %            : {pii_coverage_pct:.1f}%")
print(f"Certification coverage %  : {cert_coverage_pct:.1f}%")
print(f"CDE missing steward       : {cde_missing_steward:,}")
print(f"Metadata completeness %   : {metadata_completeness_pct:.1f}%")
print(f"Overall compliance %      : {overall_compliance_pct:.1f}%")

In [0]:
from pyspark.sql.functions import greatest

non_compliant_summary = (
    silver.filter(F.col("compliance_flag") == "NON-COMPLIANT")
    .groupBy("table_name")
    .agg(
        F.count("*").alias("non_compliant_row_count"),
        F.sum(F.col("rule01_mandatory_null").cast("int")).alias("rule01_violations"),
        F.sum(F.col("rule03_security_invalid").cast("int")).alias("rule03_violations"),
        F.sum(F.col("rule04_cert_level_invalid").cast("int")).alias("rule04_violations"),
        F.sum(F.col("rule05_pii_not_confidential").cast("int")).alias("rule05_violations"),
        F.sum(F.col("rule06_below_col_standard").cast("int")).alias("rule06_violations"),
        F.sum(F.col("rule07_cde_no_steward").cast("int")).alias("rule07_violations"),
        F.sum(F.col("rule08_invalid_count_exceeds_total").cast("int")).alias("rule08_violations"),
    )
    .withColumn("worst_rule",
        F.when(
            F.col("rule03_violations") == greatest(
                F.col("rule01_violations"), F.col("rule03_violations"),
                F.col("rule04_violations"), F.col("rule05_violations"),
                F.col("rule06_violations"), F.col("rule07_violations"),
                F.col("rule08_violations")
            ), "rule03"
        ).when(
            F.col("rule07_violations") == greatest(
                F.col("rule01_violations"), F.col("rule03_violations"),
                F.col("rule04_violations"), F.col("rule05_violations"),
                F.col("rule06_violations"), F.col("rule07_violations"),
                F.col("rule08_violations")
            ), "rule07"
        ).when(
            F.col("rule04_violations") == greatest(
                F.col("rule01_violations"), F.col("rule03_violations"),
                F.col("rule04_violations"), F.col("rule05_violations"),
                F.col("rule06_violations"), F.col("rule07_violations"),
                F.col("rule08_violations")
            ), "rule04"
        ).when(
            F.col("rule06_violations") == greatest(
                F.col("rule01_violations"), F.col("rule03_violations"),
                F.col("rule04_violations"), F.col("rule05_violations"),
                F.col("rule06_violations"), F.col("rule07_violations"),
                F.col("rule08_violations")
            ), "rule06"
        ).when(
            F.col("rule05_violations") == greatest(
                F.col("rule01_violations"), F.col("rule03_violations"),
                F.col("rule04_violations"), F.col("rule05_violations"),
                F.col("rule06_violations"), F.col("rule07_violations"),
                F.col("rule08_violations")
            ), "rule05"
        ).otherwise("rule01")
    )
    .withColumn("suggested_fix",
        F.when(F.col("worst_rule") == "rule03",
            F.lit("Fix security_classification — allowed values: Internal, Confidential, Public"))
        .when(F.col("worst_rule") == "rule07",
            F.lit("Assign a data steward to all critical data elements per GOV-111 RULE-07"))
        .when(F.col("worst_rule") == "rule04",
            F.lit("Add or fix certification_level — allowed values: Registered, Certified, Documented"))
        .when(F.col("worst_rule") == "rule06",
            F.lit("Review table structure — below the 38.6 column standard per GOV-111 RULE-06"))
        .when(F.col("worst_rule") == "rule05",
            F.lit("Reclassify PII columns as Confidential per GOV-111 RULE-05"))
        .otherwise(
            F.lit("Populate mandatory fields: column_id, table_id, column_name, table_name"))
    )
    .orderBy("non_compliant_row_count", ascending=False)
)

non_compliant_summary.display()

In [0]:
@dlt.table(
    name="gold.governance_metrics",
    comment="Gold layer — KPI summary metrics, one row"
)
def governance_metrics():
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
    from datetime import datetime

    gold_data = [(
        int(total_tables),
        int(total_columns),
        round(float(structural_compliance_pct), 2),
        round(float(pii_coverage_pct), 2),
        round(float(cert_coverage_pct), 2),
        int(cde_missing_steward),
        round(float(metadata_completeness_pct), 2),
        round(float(overall_compliance_pct), 2),
        datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    )]

    gold_schema = StructType([
        StructField("total_tables",                LongType(),   False),
        StructField("total_columns",               LongType(),   False),
        StructField("structural_compliance_pct",   DoubleType(), False),
        StructField("pii_coverage_pct",            DoubleType(), False),
        StructField("certification_coverage_pct",  DoubleType(), False),
        StructField("cde_missing_steward_count",   LongType(),   False),
        StructField("metadata_completeness_pct",   DoubleType(), False),
        StructField("overall_compliance_pct",      DoubleType(), False),
        StructField("calculated_at",               StringType(), False),
    ])

    return spark.createDataFrame(gold_data, schema=gold_schema)


@dlt.table(
    name="gold.non_compliant_tables",
    comment="Gold layer — per-table violations and suggested fixes"
)
def non_compliant_tables_table():
    return non_compliant_summary


@dlt.table(
    name="gold.table_column_counts",
    comment="Gold layer — per-table column counts with compliance status"
)
def table_column_counts():
    return (
        silver.select("table_name", "column_count", "col_standard_threshold", "rule06_below_col_standard")
              .distinct()
              .withColumnRenamed("rule06_below_col_standard", "below_column_standard")
    )